In [ ]:
import os
os.environ["OPENAI_API_KEY"]="your_openai_api_key_here"  # Replace with your actual OpenAI API key

In [ ]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 23.1 MB/s eta 0:00:00


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [ ]:
#1) Tool Creation

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [ ]:
print(multiply.invoke({'a':3, 'b':4}))
print(multiply.name)
print(multiply.description)
multiply.args

12
multiply
Given 2 numbers a and b this tool returns their product


{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [ ]:
#2) Tool Binding

In [ ]:
llm = ChatOpenAI(model="gpt-6-luna")

In [ ]:
llm_with_tools = llm.bind_tools([multiply]) #bind_tools() is a LangChain method that binds a list of tools to a chat model by providing their names, descriptions, and input schemas,
#allowing the model to decide when to request a tool call. It does not execute the tools by itself.

In [ ]:
llm_with_tools.invoke("hi! how are u doin?") # see "llm" is a runnable model configuration which we know and pass queries to it but llm_with_tools is a
# new runable model configuration that knows about the tools that's it no major change so u pass queries to it as well

#u could see tool_calls is empty

AIMessage(content=[{'type': 'text', 'text': 'Hi! I’m doing well, thanks for asking 😊 How are you?', 'annotations': [], 'id': 'msg_0c0afff505946f94006ac89a2ce18887d2966c6875d84a6469', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_0c0afff505946f94006ac89a2b766087d28bb85011ea700368', 'created_at': 1791531563.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0c0afff505946f94006ac89a2b766087d28bb85011ea700368', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 19, 'total_tokens': 79, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [ ]:
llm_with_tools.invoke("multiply 4 with 5") #but here:

AIMessage(content=[{'arguments': '{"a":4,"b":5}', 'call_id': 'call_SX1kNbWFHFST7SjiytBji6qf', 'name': 'multiply', 'type': 'function_call', 'id': 'fc_03e685805d44cfa6006ac89a42128487d28682274b6caddaf7', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_03e685805d44cfa6006ac89a41095087d28d48dd264b0ff95d', 'created_at': 1791531585.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_03e685805d44cfa6006ac89a41095087d28d48dd264b0ff95d', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 5}, 'id': 'call_SX1kNbWFHFST7SjiytBji6qf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 21, 'total_tokens': 79, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [ ]:
llm_with_tools.invoke("multiply 4 with 5").tool_calls #which is a list which can contain one or more tool calls

[{'name': 'multiply',
  'args': {'a': 4, 'b': 5},
  'id': 'call_q9bFddDUQDeyvQlX0IjeTBLp',
  'type': 'tool_call'}]

In [ ]:
#let's access the first element of the list
llm_with_tools.invoke("multiply 4 with 5").tool_calls[0] #this is the structured output or the schema we r talking about in the notes under the heading "3)Tool Calling" where name and
#arguments are generated will be written and that is what we are getting in the output below

{'name': 'multiply',
 'args': {'a': 4, 'b': 5},
 'id': 'call_pZ1rqxRJR6UQ54wXyQhgv6xj',
 'type': 'tool_call'}

In [ ]:
llm_with_tools.invoke("multiply 4 with 5").tool_calls[0]['args']

{'a': 4, 'b': 5}

In [ ]:
#3) Tool Execution
res = multiply.invoke(llm_with_tools.invoke("multiply 4 with 5").tool_calls[0]['args'])
res

20

In [ ]:
#so yes:
x = llm_with_tools.invoke("multiply 4 with 5")
x

AIMessage(content=[{'arguments': '{"a":4,"b":5}', 'call_id': 'call_eQj83lM6R0EWB3aVwsWuOc2v', 'name': 'multiply', 'type': 'function_call', 'id': 'fc_093ccd2061f68463006ac8aefb9da887d1bdd8914a9d3a32b2', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_093ccd2061f68463006ac8aefab10c87d1b72d169cad4b81f5', 'created_at': 1791536890.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_093ccd2061f68463006ac8aefab10c87d1b72d169cad4b81f5', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 5}, 'id': 'call_eQj83lM6R0EWB3aVwsWuOc2v', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 21, 'total_tokens': 79, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [ ]:
y = llm_with_tools.invoke("multiply 4 with 5").tool_calls
y

[{'name': 'multiply',
  'args': {'a': 4, 'b': 5},
  'id': 'call_fxDCdkAclOi1E2InVuSVcKTp',
  'type': 'tool_call'}]

In [ ]:
r1 = llm_with_tools.invoke("multiply 4 with 5").tool_calls[0]
r1

{'name': 'multiply',
 'args': {'a': 4, 'b': 5},
 'id': 'call_kbvgE6pJEfTodJo42CjNWb0z',
 'type': 'tool_call'}

In [ ]:
r2 = llm_with_tools.invoke("multiply 4 with 5").tool_calls[0]['args']
r2

{'a': 4, 'b': 5}

In [ ]:
multiply.invoke(r2)

20

In [ ]:
multiply.invoke(r1) #here u are passing enitre tool call of one single tool which is multiply obv
#result is a ToolMessage - u get when u execute a tool with help of a tool call.
#feature/speciality of this ToolMessage is that we can send it entirely to the llm which means that look llm! i have executed the tool and this is my response, i mean
#llm can analyse that and can give the answer, but how will u do that - by maintaining a conversational history:

ToolMessage(content='20', name='multiply', tool_call_id='call_kbvgE6pJEfTodJo42CjNWb0z')

In [ ]:
#untill now u have executed now let llm execute:

In [ ]:
query = HumanMessage("multiply 7 with 500")

In [ ]:
messages = [query] #u have created a list and have inserted query in it

In [ ]:
ai_m = llm_with_tools.invoke(messages) #u will get an ai message which has tool_call
ai_m

AIMessage(content=[{'arguments': '{"a":7,"b":500}', 'call_id': 'call_xJqA7zpkrwCMR7MoWMiNxXP8', 'name': 'multiply', 'type': 'function_call', 'id': 'fc_0414d3942f82c87b006ac8b3350fa887d1bf949b7775099859', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0414d3942f82c87b006ac8b3343c7c87d1b23238b67fd2ab8f', 'created_at': 1791537972.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0414d3942f82c87b006ac8b3343c7c87d1b23238b67fd2ab8f', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 500}, 'id': 'call_xJqA7zpkrwCMR7MoWMiNxXP8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 21, 'total_tokens': 79, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [ ]:
#let's append this to messages too
messages.append(ai_m)

In [ ]:
messages #now u have both human and ai messages

[HumanMessage(content='multiply 7 with 500', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'arguments': '{"a":7,"b":500}', 'call_id': 'call_xJqA7zpkrwCMR7MoWMiNxXP8', 'name': 'multiply', 'type': 'function_call', 'id': 'fc_0414d3942f82c87b006ac8b3350fa887d1bf949b7775099859', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0414d3942f82c87b006ac8b3343c7c87d1b23238b67fd2ab8f', 'created_at': 1791537972.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0414d3942f82c87b006ac8b3343c7c87d1b23238b67fd2ab8f', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 500}, 'id': 'call_xJqA7zpkrwCMR7MoWMiNxXP8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 21, 'total_tokens': 79, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reas

In [ ]:
llm_with_tools.invoke(messages)
#VERY IMPORTANT
#the model requested multiply(a=7, b=500) -i.e what tool call request means; but your message history did not contain the result of executing that function.
#Since you're using the OpenAI Responses API through LangChain, the API returned this error: No tool output found for function call call_xJqA7zpkrwCMR7MoWMiNxXP8 or for the 'id' just right above.

## The AI message contains a tool-call request specifying which tool to use and what arguments to pass, but it does not contain the tool's execution result.
# Passing the HumanMessage and AIMessage alone leaves the function call unresolved. WE MUST EXECUTE THE REQUESTED TOOL and append a ToolMessage containing its output, associated with the corresponding tool-call ID.
# Then, we can pass the complete message history back to the LLM so it can use the tool output to generate its final response.

OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': 'No tool output found for function call call_xJqA7zpkrwCMR7MoWMiNxXP8.', 'type': 'invalid_request_error', 'param': 'input', 'code': None}}

In [ ]:
a=messages[1].tool_calls
a

[{'name': 'multiply',
  'args': {'a': 7, 'b': 500},
  'id': 'call_xJqA7zpkrwCMR7MoWMiNxXP8',
  'type': 'tool_call'}]

In [ ]:
b=messages[1].tool_calls[0]
b

{'name': 'multiply',
 'args': {'a': 7, 'b': 500},
 'id': 'call_xJqA7zpkrwCMR7MoWMiNxXP8',
 'type': 'tool_call'}

In [ ]:
to_m = multiply.invoke(b)
to_m

ToolMessage(content='3500', name='multiply', tool_call_id='call_xJqA7zpkrwCMR7MoWMiNxXP8')

In [ ]:
messages.append(to_m)
messages

[HumanMessage(content='multiply 7 with 500', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'arguments': '{"a":7,"b":500}', 'call_id': 'call_xJqA7zpkrwCMR7MoWMiNxXP8', 'name': 'multiply', 'type': 'function_call', 'id': 'fc_0414d3942f82c87b006ac8b3350fa887d1bf949b7775099859', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0414d3942f82c87b006ac8b3343c7c87d1b23238b67fd2ab8f', 'created_at': 1791537972.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0414d3942f82c87b006ac8b3343c7c87d1b23238b67fd2ab8f', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 500}, 'id': 'call_xJqA7zpkrwCMR7MoWMiNxXP8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 21, 'total_tokens': 79, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reas

In [ ]:
llm_with_tools.invoke(messages) #an ai message obv since a response

AIMessage(content=[{'type': 'text', 'text': '7 × 500 = 3,500.', 'annotations': [], 'id': 'msg_0414d3942f82c87b006ac8bcffda2c87d1908b57889171f29b', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_0414d3942f82c87b006ac8bcfee4c087d1b831177fc9cdbf5b', 'created_at': 1791540478.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0414d3942f82c87b006ac8bcfee4c087d1b831177fc9cdbf5b', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 91, 'output_tokens': 14, 'total_tokens': 105, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [ ]:
llm_with_tools.invoke(messages).text

'7 × 500 = 3,500'

# **CURRENCY CONVERSION TOOL**

In [ ]:
import requests

In [ ]:
#1)TOOL CREATION
#tool 1
@tool
def get_conversion_factor(base_currency: str, target_currency: str)->float:
  """This function fetches the currency conversion factor between a given base currency and a target currency"""
  url = f"https://v6.exchangerate-api.com/v6/<API_KEY>/pair/{base_currency}/{target_currency}" #Replace <API_KEY> with your actual API key from the ExchangeRate API service.
  res = requests.get(url) #This sends an HTTP GET request to the API and returns a RESPONSE object from the Python requests library.
  return res.json() #.json() method parses the JSON response body and converts it into corresponding Python objects. In this example, data becomes a Python dictionary

In [ ]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'inr'}) #remember it is a tool and it is a runnable

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791590401,
 'time_last_update_utc': 'Sat, 10 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1791676801,
 'time_next_update_utc': 'Sun, 11 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.8685}

In [ ]:
#tool 2
@tool
def convert(base_currency_value: int, conversion_rate: float) -> float:
  """ given a currency conversion rate this function calculates the target currency value from a given base currency value """
  return base_currency_value * conversion_rate

In [ ]:
convert(100, 96.5) #error since it is a tool

TypeError: 'StructuredTool' object is not callable

In [ ]:
convert.invoke({'base_currency_value':10, 'conversion_rate':96.8837})

968.837

In [ ]:
#2) TOOL BINDING
llm = ChatOpenAI(model="gpt-6-luna")

In [ ]:
llm_and_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
#3)TOOL CALLING
msgs = [HumanMessage('convert 10 us dollars into indian rupees and state the conversion factor between usd and inr.')]

In [ ]:
ai_msg = llm_and_tools.invoke(msgs)
ai_msg

AIMessage(content=[{'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'call_id': 'call_VhnyH2dlYWvArmen9tHrA1zw', 'name': 'get_conversion_factor', 'type': 'function_call', 'id': 'fc_0a462a7fcb10cd53006ac9deec49d087d0bf337f6bc82e514e', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0a462a7fcb10cd53006ac9deeaa94887d09cb06e6c96690500', 'created_at': 1791614698.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0a462a7fcb10cd53006ac9deeaa94887d09cb06e6c96690500', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_VhnyH2dlYWvArmen9tHrA1zw', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 120, 'output_tokens': 26, 'total_tokens': 146, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'rea

In [ ]:
#Tool Calling -
ai_msg.tool_calls

# Only one tool appeared because the model decided that get_conversion_factor was the appropriate tool to call first to answer your question.

# Your code binds both tools:
# llm_and_tools = llm.bind_tools([get_conversion_factor, convert])
# This makes both tools available to the model; it does not instruct the model to call both.

# Your request involves two operations:
# 1. get_conversion_factor — fetch the exchange rate between INR and USD.
# 2. convert — multiply 10 INR by the exchange rate.
# The second operation depends on the result of the first. Since the exchange rate wasn't available yet, the model requested get_conversion_factor first.

#So, the reason is not that your query is wrong or that the second tool wasn't bound. The model chose a sequential approach, requesting the first tool before proceeding to the second.

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_VhnyH2dlYWvArmen9tHrA1zw',
  'type': 'tool_call'}]

In [ ]:
msgs.append(ai_msg)
msgs

[HumanMessage(content='convert 10 us dollars into indian rupees and state the conversion factor between usd and inr.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'call_id': 'call_VhnyH2dlYWvArmen9tHrA1zw', 'name': 'get_conversion_factor', 'type': 'function_call', 'id': 'fc_0a462a7fcb10cd53006ac9deec49d087d0bf337f6bc82e514e', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0a462a7fcb10cd53006ac9deeaa94887d09cb06e6c96690500', 'created_at': 1791614698.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0a462a7fcb10cd53006ac9deeaa94887d09cb06e6c96690500', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_VhnyH2dlYWvArmen9tHrA1zw', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

In [ ]:
re1 = ai_msg.tool_calls[0]
re1

{'name': 'get_conversion_factor',
 'args': {'base_currency': 'USD', 'target_currency': 'INR'},
 'id': 'call_VhnyH2dlYWvArmen9tHrA1zw',
 'type': 'tool_call'}

In [ ]:
#now u should not directly use name of the tool to generate a tool message since we r developing kind an app so we should not do internal working , we should let the program and llm
# to take care of everything
#so first let's create a dict where u map the tools to their own names ('name' - below check the key)
get_conversion_factor

StructuredTool(name='get_conversion_factor', description='This function fetches the currency conversion factor between a given base currency and a target currency', args_schema=<class 'langchain_core.utils.pydantic.get_conversion_factor'>, func=<function get_conversion_factor at 0x7b5996c4bba0>)

In [ ]:
convert

StructuredTool(name='convert', description='given a currency conversion rate this function calculates the target currency value from a given base currency value', args_schema=<class 'langchain_core.utils.pydantic.convert'>, func=<function convert at 0x7b5996aaf920>)

In [ ]:
tool_by_name = {
    get_conversion_factor.name : get_conversion_factor, #tool's name = actual tool obj , so tool is being stored in its name
    convert.name : convert #(sinec convert is an obj we are able to access its attribute with . operator) convert tool is being stored in name of convert.
}
print(tool_by_name.keys()) #check whether the keys are names of the tools

dict_keys(['get_conversion_factor', 'convert'])


In [ ]:
re1['name']

'get_conversion_factor'

In [ ]:
tool_name = re1['name'] #can u write tool_name = get_conversion_factor.name ? if u did then how did u know that the first tool llm took is get_conversion name itself?
# so u shouldn't know internal work . # u have the tool name of the tool the llm chose which u don't know
#now tool_by_name dict has actual tools which are mapped to their own names. now what if i give the name from re1 as key to the dict so that i get the actual tool :
tool = tool_by_name[tool_name]
tool

StructuredTool(name='get_conversion_factor', description='This function fetches the currency conversion factor between a given base currency and a target currency', args_schema=<class 'langchain_core.utils.pydantic.get_conversion_factor'>, func=<function get_conversion_factor at 0x7b5996c4bba0>)

In [ ]:
#so now use the actual tool for invoking or generating a tool message: #if u only pass re1['args'] u shall get the result but we need tool message so that we can append to msgs
#but tool message is obtained only by passing entire tool_call request: which is entire re1
#TOOL EXECUTION - FIRST TOOL
t_msg = tool.invoke(re1)
t_msg

ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1791590401, "time_last_update_utc": "Sat, 10 Oct 2026 00:00:01 +0000", "time_next_update_unix": 1791676801, "time_next_update_utc": "Sun, 11 Oct 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 96.8685}', name='get_conversion_factor', tool_call_id='call_VhnyH2dlYWvArmen9tHrA1zw')

In [ ]:
msgs.append(t_msg)
msgs

[HumanMessage(content='convert 10 us dollars into indian rupees and state the conversion factor between usd and inr.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'call_id': 'call_VhnyH2dlYWvArmen9tHrA1zw', 'name': 'get_conversion_factor', 'type': 'function_call', 'id': 'fc_0a462a7fcb10cd53006ac9deec49d087d0bf337f6bc82e514e', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0a462a7fcb10cd53006ac9deeaa94887d09cb06e6c96690500', 'created_at': 1791614698.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0a462a7fcb10cd53006ac9deeaa94887d09cb06e6c96690500', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_VhnyH2dlYWvArmen9tHrA1zw', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

In [ ]:
#LLM GIVING RESPONSE OF TOOL1:
ai_msg_new = llm_and_tools.invoke(msgs)
ai_msg_new

AIMessage(content=[{'arguments': '{"base_currency_value":10,"conversion_rate":96.8685}', 'call_id': 'call_wPvwMEyaSZGDujTq446ll9b7', 'name': 'convert', 'type': 'function_call', 'id': 'fc_0a462a7fcb10cd53006ac9e0570c0c87d0b2c079a94ee8d3aa', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0a462a7fcb10cd53006ac9e055f2a887d0ab9a8221bd9ab284', 'created_at': 1791615062.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0a462a7fcb10cd53006ac9e055f2a887d0ab9a8221bd9ab284', tool_calls=[{'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 96.8685}, 'id': 'call_wPvwMEyaSZGDujTq446ll9b7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 298, 'output_tokens': 27, 'total_tokens': 325, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [ ]:
msgs.append(ai_msg_new)
msgs

[HumanMessage(content='convert 10 us dollars into indian rupees and state the conversion factor between usd and inr.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'call_id': 'call_VhnyH2dlYWvArmen9tHrA1zw', 'name': 'get_conversion_factor', 'type': 'function_call', 'id': 'fc_0a462a7fcb10cd53006ac9deec49d087d0bf337f6bc82e514e', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0a462a7fcb10cd53006ac9deeaa94887d09cb06e6c96690500', 'created_at': 1791614698.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0a462a7fcb10cd53006ac9deeaa94887d09cb06e6c96690500', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_VhnyH2dlYWvArmen9tHrA1zw', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

In [ ]:
ai_msg_new.tool_calls

[{'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_rate': 96.8685},
  'id': 'call_wPvwMEyaSZGDujTq446ll9b7',
  'type': 'tool_call'}]

In [ ]:
re2= ai_msg_new.tool_calls[0]
re2

{'name': 'convert',
 'args': {'base_currency_value': 10, 'conversion_rate': 96.8685},
 'id': 'call_wPvwMEyaSZGDujTq446ll9b7',
 'type': 'tool_call'}

In [ ]:
tool2_name = re2['name']
tool2 = tool_by_name[tool2_name]
tool2

StructuredTool(name='convert', description='given a currency conversion rate this function calculates the target currency value from a given base currency value', args_schema=<class 'langchain_core.utils.pydantic.convert'>, func=<function convert at 0x7b5996aaf920>)

In [ ]:
#TOOL EXECUTION OF TOOL 2:
t_msg_new = tool2.invoke(re2)
t_msg_new

ToolMessage(content='968.685', name='convert', tool_call_id='call_wPvwMEyaSZGDujTq446ll9b7')

In [ ]:
msgs.append(t_msg_new)
msgs

[HumanMessage(content='convert 10 us dollars into indian rupees and state the conversion factor between usd and inr.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'call_id': 'call_VhnyH2dlYWvArmen9tHrA1zw', 'name': 'get_conversion_factor', 'type': 'function_call', 'id': 'fc_0a462a7fcb10cd53006ac9deec49d087d0bf337f6bc82e514e', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0a462a7fcb10cd53006ac9deeaa94887d09cb06e6c96690500', 'created_at': 1791614698.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0a462a7fcb10cd53006ac9deeaa94887d09cb06e6c96690500', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_VhnyH2dlYWvArmen9tHrA1zw', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

In [ ]:
llm_and_tools.invoke(msgs)

AIMessage(content=[{'type': 'text', 'text': 'The conversion factor is **1 USD = 96.8685 INR**.  \nSo **10 USD = ₹968.685**, or about **₹968.69**.', 'annotations': [], 'id': 'msg_0a462a7fcb10cd53006ac9e2efe7ec87d09d89241f84cb0708', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_0a462a7fcb10cd53006ac9e2eee9d487d0b1aff64d5b77ff6c', 'created_at': 1791615727.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_0a462a7fcb10cd53006ac9e2eee9d487d0b1aff64d5b77ff6c', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 338, 'output_tokens': 40, 'total_tokens': 378, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [ ]:
#FINAL ANSWER:
llm_and_tools.invoke(msgs).text

'The conversion factor is **1 USD = 96.8685 INR**.  \nSo, **10 USD = ₹968.69** (approximately).'

In [ ]:
#ABOVE PIECES OF CODE IS LIKE DRY RUN WHAT IF U USE A TOOL-EXECUTION LOOP:
#2.1_CURRENCY_CONVERSION_TOOL_LANGCHAIN.IPYNB FILE